# Comparacion de modelos de riesgo

Este notebook compara el modelo Negative Binomial contra Poisson y un baseline usando el flujo activo desde CSV.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_poisson_deviance, mean_squared_error

repo_root = Path.cwd()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.model import (
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    build_temporal_training_table,
    predict_counts,
    train_count_model,
)

csv_path = repo_root / 'accidentes_con_trafico_final.csv'

csv_path

In [ ]:
table = build_temporal_training_table(csv_path, max_streets=100)

train = table[table['split'] == 'train'].copy()
validation = table[table['split'] == 'validation'].copy()
test = table[table['split'] == 'test'].copy()

len(train), len(validation), len(test)

In [ ]:
def metrics(model_name, split_name, y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 1e-9, None)
    return {
        'model': model_name,
        'split': split_name,
        'mean_poisson_deviance': mean_poisson_deviance(y_true, y_pred),
        'mae': mean_absolute_error(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
    }


rows = []
targets = {'validation': validation, 'test': test}

for split_name, target in targets.items():
    baseline = np.full(len(target), train['accident_count'].mean())
    rows.append(metrics('global_baseline', split_name, target['accident_count'], baseline))

models = [
    ('poisson_with_weather', 'poisson', CATEGORICAL_FEATURES, NUMERIC_FEATURES),
    ('negative_binomial_without_weather', 'negative_binomial', ['temporal_bin_4h'], NUMERIC_FEATURES),
    ('negative_binomial_with_weather', 'negative_binomial', CATEGORICAL_FEATURES, NUMERIC_FEATURES),
]

for name, family, categorical, numeric in models:
    model = train_count_model(train, categorical, numeric, family)
    for split_name, target in targets.items():
        rows.append(metrics(name, split_name, target['accident_count'], predict_counts(model, target)))

results = pd.DataFrame(rows)
results.sort_values(['split', 'mean_poisson_deviance'])